# AdventureWorks — Contribution Margin & Markup Simulation

**Group 3 · Business Reports and Dashboards · FH Joanneum**

Management believes that a few customer segments, products and regions generate most of the contribution margin (CM),
and is considering **additional markups** on (a) low-CM products and (b) orders shipped to geographically distant
regions. This notebook walks through the full analysis:

1. Data extraction & cleaning
2. Exploratory analysis
3. Contribution margin per order line (with / without freight)
4. Pareto / ABC analysis
5. Definition of "distant region"
6. Price sensitivity model
7. Customer churn model
8. Markup scenario simulation
9. Outputs & limitations

All logic lives in `src/` (one module per step); settings are in `config.yaml`; modelling assumptions are documented in
`ASSUMPTIONS.md` and output schemas in `SCHEMA.md`. `python -m src.pipeline` runs the same steps and writes the CSVs.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd

from src.config import load_config
from src import cleaning, cm, segments, pareto, distance, elasticity, churn, scenarios

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# validated categorical palette (fixed order) and ABC ramp
C = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
CH = {"Online": C[0], "Reseller": C[1]}
plt.rcParams.update({"figure.figsize": (10, 4.2), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "axes.grid.axis": "y", "lines.linewidth": 2,
                     "font.size": 10})
money = mtick.FuncFormatter(lambda x, _: f"${x/1e6:,.1f}M")

cfg = load_config()
FIG = cfg["paths"]["figures_dir"]
def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIG / f"{name}.png", dpi=160)

## 1. Data extraction & cleaning

**Source:** Microsoft AdventureWorks2022 (OLTP), restored into SQL Server 2022 in Docker. `src/extract.py` flattens
15 tables into `data/raw/*.csv` (order header/detail, product hierarchy, cost & list price history, customers, store and
person demographics, territories, addresses with coordinates, special offers, ship methods).

`src/cleaning.py` runs every check below and logs it: duplicates, primary keys, referential integrity, valid
quantities/prices/discounts, LineTotal and SubTotal consistency, channel coverage, history-table semantics, address
coordinates, and missing categories.

In [ ]:
raw = cleaning.load_raw(cfg["paths"]["raw_dir"])
tables, log = cleaning.clean(raw)
print({k: len(v) for k, v in tables.items()})
log

**Findings**
- The data is structurally clean: no duplicates, no orphan keys, LineTotal matches the price formula.
- **918 addresses (France, Germany, UK) have coordinates (0, 0)** → imputed with the median of the same city.
- `ProductCostHistory` / `ProductListPriceHistory` end dates are **inclusive**; resellers kept selling 25 discontinued
  products after their last cost period → the last known cost is used (flagged in `cost_within_period`).
- **Reseller data ends on 2014-05-01, online data on 2014-06-30** → every baseline is measured per channel.

## 2. Exploratory analysis

In [ ]:
fact = cm.build_fact(tables, cfg)
customers = segments.build_customers(fact, tables)

m = fact.groupby(["OrderMonth", "Channel"]).revenue.sum().unstack()
fig, ax = plt.subplots()
for ch in m:
    ax.plot(m.index, m[ch], color=CH[ch], label=ch)
ax.yaxis.set_major_formatter(money); ax.set_title("Monthly revenue by channel"); ax.legend(frameon=False)
save(fig, "eda_monthly_revenue")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fact.groupby("Channel").price_index.plot.hist(bins=40, alpha=0.7, ax=axes[0], legend=True,
                                              color=[CH["Online"], CH["Reseller"]])
axes[0].set_title("Price paid / list price"); axes[0].set_xlabel("price index")

opc = customers.groupby("CustomerType").orders.value_counts(normalize=True).rename("share").reset_index()
for i, (t, g) in enumerate(opc.groupby("CustomerType")):
    g = g[g.orders <= 12]
    axes[1].bar(g.orders + (i - 0.5) * 0.4, g.share, width=0.38, color=C[i], label=t)
axes[1].set_title("Orders per customer"); axes[1].legend(frameon=False); axes[1].set_xlabel("orders")

coh = customers.groupby(customers.first_order.dt.to_period("Q")).size()
axes[2].bar(coh.index.astype(str), coh.values, color=C[0]); axes[2].set_title("New customers per quarter")
axes[2].tick_params(axis="x", rotation=90)
save(fig, "eda_price_orders_cohorts")
print("Online customers with exactly one order:", f"{(customers[customers.CustomerType=='Individual'].orders==1).mean():.0%}")

**EDA takeaways**
- Online customers always pay **exactly list price** and buy **one unit per line**; resellers pay **~60% of list**
  (plus occasional promotions) → there is very little price variation to learn demand response from (see §6).
- Most online customers buy once; resellers order every ~3 months.
- **Mid-2013 structural break:** new online product lines (accessories, clothing, touring bikes) and a jump in new
  customers. Models are evaluated out-of-time across this break.

## 3. Contribution margin per order line

| | Definition |
|---|---|
| Revenue | `LineTotal` = UnitPrice × OrderQty × (1 − UnitPriceDiscount) |
| COGS | OrderQty × StandardCost **valid at the order date** (ProductCostHistory) |
| CM1 | Revenue − COGS |
| Freight allocation | order Freight × line revenue / order revenue |
| CM2 | CM1 − allocated freight |

`cm.freight_allocation` in `config.yaml` chooses which one is used downstream (default: CM2). Using today's
`Product.StandardCost` instead of the historical cost would understate reseller CM by ~$2.8M.

In [ ]:
cm_tab = cm.cm_summary(fact)
cm_tab

In [ ]:
cc = fact.groupby(["Category", "Channel"]).agg(revenue=("revenue", "sum"), cm=("cm", "sum")).reset_index()
cc["cm_margin"] = cc.cm / cc.revenue
pv = cc.pivot(index="Category", columns="Channel", values="cm")
fig, ax = plt.subplots()
x = np.arange(len(pv))
for i, ch in enumerate(pv):
    ax.bar(x + (i - 0.5) * 0.38, pv[ch], width=0.36, color=CH[ch], label=ch)
ax.set_xticks(x, pv.index); ax.yaxis.set_major_formatter(money); ax.axhline(0, color="gray", lw=0.8)
ax.set_title("CM by category and channel"); ax.legend(frameon=False)
save(fig, "cm_category_channel")
cc.pivot(index="Category", columns="Channel", values="cm_margin")

**Key result:** the online channel earns almost all of the margin. **Reseller bikes are sold at roughly cost**
(reseller price ≈ 60% of list ≈ standard cost), so the reseller channel is slightly negative after freight
while carrying ~73% of revenue.

## 4. Pareto / ABC analysis

Entities are ranked by CM. Classes use the cumulative share of **positive** CM: **A** = first 80%, **B** = next 15%,
**C** = remaining positive, **Negative** = CM < 0 (kept separate because loss-makers would otherwise distort the
cumulative curve). Concentration is reported against positive and against net CM.

In [ ]:
P = pareto.run_all(fact, customers, cfg)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
for ax, dim in zip(axes, ["customer", "product"]):
    g = P[dim]
    ax.plot(g.entity_share, g.cum_positive_cm_share, color=C[0], label="share of positive CM")
    ax.plot(g.entity_share, g.cum_cm_share, color=C[1], label="share of net CM")
    ax.axhline(1, color="gray", lw=0.8, ls=":"); ax.axvline(0.2, color="gray", lw=0.8, ls=":")
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
    ax.set_title(f"Pareto curve — {dim}s"); ax.set_xlabel(f"share of {dim}s (ranked by CM)"); ax.legend(frameon=False)
save(fig, "pareto_customer_product")

for dim in ["customer", "product"]:
    c = pareto.concentration(P[dim]).set_index("top_share")
    print(f"Top 20% of {dim}s generate {c.loc[0.2, 'cm_share_of_positive']:.0%} of positive CM "
          f"({c.loc[0.2, 'cm_share_of_net']:.0%} of net CM)")
pd.concat({d: pareto.class_summary(P[d]).set_index("abc_class") for d in ["customer", "product"]})

In [ ]:
for dim in ["segment", "territory", "category", "channel_category", "store_business_type", "rfm_segment"]:
    display(P[dim].drop(columns=["lines", "rank", "entity_share", "cogs"]).round(3))

**Management's hypothesis holds, with a twist:**
- CM is highly concentrated: about a fifth of customers and products generate the large majority of positive CM.
- The "many customers that contribute little" are mostly one-time online buyers (class C); the real problem is
  **~400 store customers with negative CM** who account for almost half of revenue.
- Individual customers have a near-identical ~38% margin regardless of income band — segment differences are driven
  by **channel**, not demographics.
- Australia is the most profitable territory; the three eastern/central US territories are slightly negative (reseller-heavy).

## 5. Distance definition

Three options were compared:

| Option | Pros | Cons |
|---|---|---|
| Territory group ≠ North America | simple, matches sales organisation | coarse; Canada counts as "near" even for Toronto, US East Coast as near |
| **Great-circle distance from HQ (Bothell, WA)** to the ship-to address | continuous, uses the actual delivery address, can be banded and thresholded | needs coordinates (available for all addresses after cleaning) |
| Freight per unit | reflects shipping cost | **freight is a flat % of order value** in AdventureWorks, unrelated to distance |

In [ ]:
cmp, corr = distance.compare_definitions(fact, cfg)
print({k: round(v, 3) for k, v in corr.items()})
order = fact.groupby("SalesOrderID").agg(d=("distance_km", "first"), fr=("freight_alloc", "sum"),
                                         rev=("revenue", "sum"), ch=("Channel", "first"))
fig, ax = plt.subplots()
for ch, g in order.groupby("ch"):
    s = g.sample(min(len(g), 3000), random_state=1)
    ax.scatter(s.d, s.fr / s.rev, s=10, alpha=0.4, color=CH[ch], label=ch)
ax.set_xlabel("distance from HQ (km)"); ax.set_ylabel("freight / revenue")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.legend(frameon=False)
ax.set_title("Freight does not increase with distance")
save(fig, "distance_freight")
cmp

**Decision: great-circle distance, distant = more than 5,000 km from HQ** (configurable). At this threshold the flag
equals Europe + Pacific, but the continuous distance also supports distance bands and threshold sensitivity.
**Implication for management:** because freight is charged as a percentage of order value, a distant-region markup is
pure price discrimination, not cost recovery.

## 6. Price sensitivity model

AdventureWorks has no price experiments. Demand response is estimated from the price variation that exists, on a
product × territory × channel × month panel (volume-discount lines excluded, since the discount is triggered by the
quantity):

- **Fixed-effects log-log OLS** (per channel): log(units) ~ log(price) + price×category + price×region interactions
  + product×territory and month fixed effects → identified only from price changes **within** a product over time.
- **LightGBM** on log(units) with the price index and controls; elasticity = response to a ±5% price change.

An estimate is used only if it is significant (p < 0.05), inside the plausible range [−4, −0.2] and based on
≥ 200 cells; otherwise the cell falls back to the clearly labelled **assumed** elasticity (low −0.5 / mid −1.5 /
high −2.5), which is treated as a sensitivity parameter in the scenarios.

In [ ]:
E, ediag = elasticity.estimate(fact, cfg)
display(ediag["within_variation"])
print(ediag["ols"])
E.round(3)

In [ ]:
e = E.assign(cell=E.Channel + " · " + E.Category + " · " + E.TerritoryGroup).sort_values("elasticity_ols")
fig, ax = plt.subplots(figsize=(10, 6))
y = np.arange(len(e))
lo, hi = cfg["elasticity"]["plausible_range"]
ax.axvspan(lo, hi, color="gray", alpha=0.12, label="plausible range")
ax.errorbar(e.elasticity_ols, y, xerr=1.96 * e.se_ols, fmt="o", color=C[0], label="FE OLS (95% CI)")
ax.scatter(e.elasticity_lgbm, y, marker="s", color=C[1], label="LightGBM", zorder=3)
ax.set_yticks(y, e.cell); ax.axvline(0, color="gray", lw=0.8); ax.set_xlabel("elasticity")
ax.grid(axis="x", alpha=0.25); ax.legend(frameon=False, loc="lower right")
ax.set_title("Elasticity estimates vs plausible range")
save(fig, "elasticity_estimates")
print("Credible cells:", int(E.plausible.sum()), "of", len(E))

**Conclusion: the data cannot support credible elasticities.**
- Online prices only change at the yearly list-price updates, which coincide with the mid-2013 assortment change →
  OLS gives implausibly elastic, mostly insignificant values.
- Reseller unit prices are **tiered by quantity** (larger orders get a lower price ratio) → LightGBM learns
  "low price ↔ large order" (reverse causality) and returns strongly negative values; OLS within-product estimates are
  ≈ 0 or even positive (clothing).
- Only one cell (reseller accessories, North America ≈ −1.4) passes all checks, close to the assumed mid value.

→ The scenarios use the **assumed elasticity range (−0.5 / −1.5 / −2.5) as a sensitivity parameter**; "estimated"
means the credible cell plus mid elsewhere.

## 7. Customer churn model

- **Churn definition:** a customer who bought on or before the snapshot date and places **no order in the next 6
  months**.
- **Features (as of the snapshot only):** recency, frequency, monetary, CM and CM margin, tenure, order value,
  product mix (category shares, # products), price index & discount share, recent activity (90/180 days), territory,
  distance, customer type, demographics (individuals) and store survey attributes (stores).
- **Out-of-time evaluation:** train on snapshot 2013-06-30, test on 2013-12-31.
- **Final model:** refit on both snapshots, scores every customer at the end of the data (each channel at its own last
  date) → probability of not buying in the next 6 months.

In [ ]:
R = churn.train_and_evaluate(fact, customers, cfg)
print("train", R["train_size"], f"(churn {R['train_base_rate']:.0%})", "| test", R["test_size"],
      f"(churn {R['test_base_rate']:.0%}) | selected:", R["best_model"])
R["metrics"].round(3)

In [ ]:
fi = R["permutation_importance"].head(12).iloc[::-1]
sc_ = R["scored"].merge(customers[["CustomerID", "CustomerType"]])
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].barh(fi.feature, fi.permutation_importance_auc, xerr=fi.permutation_std, color=C[0])
axes[0].set_title("LightGBM permutation importance (ROC-AUC drop)"); axes[0].grid(axis="x", alpha=0.25)
for i, (t, g) in enumerate(sc_.groupby("CustomerType")):
    axes[1].hist(g.churn_prob, bins=40, alpha=0.7, color=C[i], label=t, density=True)
axes[1].set_title("Current churn probability"); axes[1].legend(frameon=False)
save(fig, "churn_importance_distribution")
R["lr_coefficients"].head(10)

**Interpretation**
- LightGBM beats the logistic baseline (ROC-AUC ≈ 0.70 overall). Stores are highly predictable (≈ 0.97: a store
  that misses its ordering cycle has churned); individuals are harder (≈ 0.67) because most buy only once.
- Strongest drivers: CM margin and product mix (accessory buyers come back for more), recency, monetary value,
  territory and distance.
- The churn model has **no price feature with variation** (no customer ever saw a price increase), so it provides the
  **baseline** risk per customer; the markup effect on churn is added by the scenario engine (§8).

## 8. Markup scenario simulation

`src/scenarios.py` applies a markup to the affected lines of the **baseline year** (last 12 months per channel):

- **Price:** affected lines × (1 + m); a line that is both low-CM and distant gets both markups.
- **Volume:** × (1 + m)^ε, with ε from the elasticity scenario per channel × category × region.
- **Freight** scales with order value (it is charged as a % of order value).
- **Customers lost:** for each customer, the revenue-weighted volume loss on their marked-up purchases v is turned
  into extra churn Δp = (1 − p₀) · κ · v, where p₀ is the churn model's baseline probability and κ (churn link,
  default 0.5) the share of the volume loss that happens by customers leaving entirely.
- **Spillover:** lost customers also stop buying the products that were *not* marked up.
- **Acceptable loss:** at most 5% of the customers we would otherwise keep.

Grid: 3 markup types × 0–20% in 2-point steps × 2 low-CM definitions × 4 elasticity scenarios.

In [ ]:
eng = scenarios.ScenarioEngine(fact, P["product"], E, R["scored"], cfg)
seg, total = eng.run_grid()
opt = scenarios.optimum(total, cfg)
print("Baseline year: revenue", f"${eng.lines.revenue.sum()/1e6:,.1f}M", "| CM", f"${eng.lines.cm.sum()/1e6:,.2f}M",
      "| customers", f"{len(eng.p0):,}")
opt.round(3)

In [ ]:
LBL = {"low_cm": "Low-CM products", "distant": "Distant regions", "both": "Both"}
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
mid = total[(total.elasticity_scenario == "mid") & total.low_cm_rule.isin(["abc_c", "n/a"])]
for i, (t, g) in enumerate(mid.groupby("markup_type", sort=False)):
    axes[0].plot(g.markup_level, g.cm_change, marker="o", ms=4, color=C[i], label=LBL[t])
    axes[1].plot(g.markup_level, g.customers_lost_pct, marker="o", ms=4, color=C[i], label=LBL[t])
axes[1].axhline(cfg["scenarios"]["max_customer_loss_pct"], color="#e34948", ls="--", lw=1, label="acceptable limit")
axes[0].yaxis.set_major_formatter(money); axes[0].set_title("CM change (mid elasticity, low CM = class C)")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[1].set_title("Customers lost")
for ax in axes:
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_xlabel("markup level"); ax.legend(frameon=False)
save(fig, "scenario_curves")

In [ ]:
# sensitivity: recommended markup level per elasticity scenario
opt.pivot_table(index=["markup_type", "low_cm_rule"], columns="elasticity_scenario",
                values=["recommended_level", "recommended_cm_change"]).round(2)

In [ ]:
# where does the extra CM come from? (mid elasticity, 10% markup on both)
s = seg[(seg.scenario_id == "both|abc_c|mid|0.10")]
s[s.dimension.isin(["Channel", "TerritoryGroup", "Category"])][
    ["dimension", "dimension_value", "cm_base", "cm", "cm_change", "affected_revenue_share", "customers_lost_pct"]].round(3)

**Scenario findings**
- **Profit rises over the whole 0–20% range in every scenario.** Reseller prices are close to cost, and with a constant
  elasticity the profit-maximising price lies far above today's (3× cost at ε = −1.5). Within the analysed range,
  profit is not the binding constraint — **customer loss is**.
- **Low-CM markups** (class C / negative products) add CM with the least customer damage: the limit is not reached
  within 20% for mid elasticity.
- **Distant-region markups** hit online customers in Europe and Australia, who make up a large share of the
  expected active base → the 5% loss limit is reached around 14–16% (mid elasticity).
- **Both combined** gives the largest CM gain but crosses the loss limit at ~10%; the recommended level is ~8%.
- Under high elasticity (−2.5) all recommended levels roughly halve; under low elasticity (−0.5) no limit is reached.
- Spillover is small and partly *positive*: customers who leave also stop buying loss-making reseller bikes.

Use the Streamlit dashboard (`http://localhost:8501`) to explore levels beyond 20%, custom elasticities and other
churn-link assumptions.

## 9. Outputs & limitations

**Outputs** (`outputs/`, schemas in `SCHEMA.md`): order-line CM fact table, product and customer dimensions (ABC class,
low-CM flags, churn probability), long Pareto/ABC table, concentration and class summaries, distance comparison,
elasticity table, churn metrics/importance, scenario results (one row per scenario × segment), scenario summary and
optimum, cleaning log. Figures for the presentation are saved in `outputs/figures/`.

**Key limitations** (details in `ASSUMPTIONS.md`)
1. Elasticities are **assumed**, not estimated — the scenario results are conditional on the elasticity range.
2. The churn uplift from markups (churn link κ) is an assumption; the churn model itself only knows baseline risk.
3. AdventureWorks is synthetic: sharp structural breaks (mid-2013), fixed reseller price ratios and freight as a
   flat % of value are artefacts of how the data was generated.
4. Constant-elasticity demand is a local approximation; large markups (> 20%) are extrapolations.
5. Competitor reactions, cannibalisation between products and reseller contract terms are not modelled.

In [ ]:
import os
print(sorted(os.listdir(cfg["paths"]["output_dir"])))
print(sorted(os.listdir(FIG)))